# EDA — Business Entity Resolution

Exploratory look at noise patterns across the three sources before building blocking/matching. See `../../CLAUDE.md` for the full problem spec.

Run this yourself — nothing here is executed by Claude.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so `import src...` works from notebooks/

import pandas as pd
from src import config
from src.io_utils import read_source, read_ground_truth
from src.blocking import prepare_records

pd.set_option('display.max_colwidth', 120)

## Load raw data

Full files are large (millions of rows). Start with `nrows=` while iterating, drop it for the real pass.

In [ ]:
s1 = read_source(config.TRAIN_SOURCE1)
s2 = read_source(config.TRAIN_SOURCE2)
s3 = read_source(config.TRAIN_SOURCE3)
gt = read_ground_truth(config.TRAIN_GROUND_TRUTH)

print(f'source1={len(s1):,} source2={len(s2):,} source3={len(s3):,} ground_truth={len(gt):,}')
s1.head()

## Country distribution

Train only has US/India. Confirm test additionally has France — the pipeline must not special-case country strings anywhere.

In [ ]:
print('train source1 countries:')
print(s1['country'].value_counts())

test_s1 = read_source(config.TEST_SOURCE1)
print('\ntest source1 countries:')
print(test_s1['country'].value_counts())

## Ground truth: match-count and singleton distribution

How many Source 1 entities have 0 / 1 / many matches? Singletons count fully toward the F_0.5 macro-average, so it's worth knowing the base rate.

In [ ]:
match_counts = gt['matched_entity_ids'].apply(len)
print(match_counts.value_counts().sort_index().head(20))
print(f"\nsingleton rate: {(match_counts == 0).mean():.1%}")

## Name/address noise patterns

Look for the noise the brief calls out: legal-suffix variants, punctuation, transliteration (non-ASCII names), stray leading/trailing junk characters, address component reordering.

In [ ]:
import re

def non_ascii_share(df):
    return df['business_name'].str.contains(r'[^\x00-\x7F]', regex=True).groupby(df['country']).mean()

print('Non-ASCII business_name share by country (source2):')
print(non_ascii_share(s2))
print('\nNon-ASCII business_name share by country (source3):')
print(non_ascii_share(s3))

In [ ]:
# Sample a few matched triples to eyeball the noise directly.
sample_gt = gt[gt['matched_entity_ids'].apply(len) > 0].sample(10, random_state=0)
for row in sample_gt.itertuples(index=False):
    s1_row = s1[s1['entity_id'] == row.source1_entity_id].iloc[0]
    print('S1:', s1_row['business_name'], '|', s1_row['business_address'], '|', s1_row['country'])
    for mid in row.matched_entity_ids:
        pool = s2 if mid.startswith('S2-') else s3
        m = pool[pool['entity_id'] == mid]
        if len(m):
            r = m.iloc[0]
            print('  ->', mid, ':', r['business_name'], '|', r['business_address'])
    print()

## Normalized-name sanity check

Confirm `prepare_records` (transliteration + legal-suffix stripping) is doing something sensible on a sample.

In [ ]:
s1_norm = prepare_records(s1.sample(2000, random_state=0))
s1_norm[['business_name', 'name_clean', 'name_core']].head(15)

## Next steps

- Run `python -m src.build_candidates --split train` then `--split test` (blocking).
- Run `python -m src.build_features --split train` then `--split test`.
- Train the matcher in `02_train_matcher.ipynb`.
- Run `python -m src.predict` to produce `output/matching_results.tsv`.
- Validate: see the command in `../../CLAUDE.md` / the official README.